# 02 · How a backtest can lie

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kolyamkl/qm-fintech-quant-workshop/blob/main/notebooks/02_backtest_honesty.ipynb)

*QM FinTech Society quant workshop · Led by **Adnan Salehin** (Lead SWE & VP, JPMorgan)*

A backtest is only as honest as the person reading it. This notebook shows the ways a result can fool you, using **strategies that have no skill at all**, so we know any good-looking number is luck, a cheat or a bug. Real strategies come in session 5, and by then you will know what to check.

**What you'll do (~60 min):**
1. Catch a strategy that **peeks at the future**
2. Watch **costs** destroy a strategy that trades too often
3. See why a handful of trades proves almost nothing
4. Learn why you **train on one period and test on another**
5. See how **trying many things** manufactures luck
6. Test fairly with a **walk-forward**
7. Finish with seven questions to ask of any backtest

Cells marked **✏️ Try it** are for you to edit and re-run.

> **Running in Google Colab?** If Colab warns that this notebook wasn't written by Google, click **Run anyway**. Then use **File → Save a copy in Drive** so your changes are kept.

## Setup

Run this cell first. In Google Colab it downloads the workshop code and price data from GitHub; on a laptop it simply finds them.

In [1]:
import os, subprocess, sys

REPO = "qm-fintech-quant-workshop"
if "google.colab" in sys.modules and os.path.basename(os.getcwd()) != REPO:
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", f"https://github.com/kolyamkl/{REPO}.git"], check=True)
    os.chdir(REPO)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_colwidth", 120)

We use SPY throughout. `buy and hold` is the benchmark from notebook 01.

In [2]:
import numpy as np
from backtester import load_prices, run_backtest

spy = load_prices("SPY")
daily = spy["Close"].pct_change().fillna(0)
buy_and_hold = run_backtest(spy, pd.Series(1, index=spy.index), cost_bps=0)
print(f"Buy and hold made {buy_and_hold.metrics['Total return']:,.0%} with a Sharpe ratio of {buy_and_hold.metrics['Sharpe ratio']:.2f}")

Buy and hold made 361% with a Sharpe ratio of 0.89


## 1. Look-ahead bias: using a price you could not have known

Here is a strategy that cannot be traded: *"hold SPY on the days it goes up."* You only know a day went up after the close, so you can never act on it.

`peek_positions` builds that signal. We then work out its total return twice. The cheating way applies each day's signal to **that same day's** return. The honest way is what `run_backtest` does: a signal calculated at today's close can only earn **tomorrow's** return, so positions are shifted by one day.

In [3]:
from backtester.honesty import peek_positions

peek = peek_positions(spy)
cheating = (1 + peek * daily).prod() - 1
honest = run_backtest(spy, peek, cost_bps=0).metrics["Total return"]
print(f"Cheating total return: {cheating:,.0%}")
print(f"Honest total return:   {honest:,.0%}")

Cheating total return: 3,953,212%
Honest total return:   57%


The cheating number is absurd because the strategy secretly knew the future. The line that stops this is in `backtester/engine.py`, marked *"THE MOST IMPORTANT LINE IN THIS REPO"*: `held = signal.shift(1)`.

Real look-ahead bugs are quieter than this one. Using a window centred on today (so it includes tomorrow's prices), scaling by the average of the whole history, or using a data column that was only filled in later are all leaks. They do not always make a result look great. They make it **meaningless**. The rule is simple: **if a number was not known at the close you decided on, you may not use it.**

**Try it:** the engine's shift is what saves the honest version. What does the honest total return become if you delete `shift(1)` from your own calculation?

## 2. Costs: every trade has a price

Every trade costs money: a fee, plus the gap between the price you could buy at and sell at. We measure it in **basis points** (bp). 1 bp is 0.01%, so 5 bp is 0.05%.

A **coin-flip strategy** holds SPY on a day if a coin lands heads and sits in cash if it lands tails. It has no skill, so before costs it should earn roughly half of what buy and hold earns, because it is in the market about half the time. It also trades on about half of all days. Watch what the cost does:

In [4]:
from backtester.honesty import coin_flip_positions

flips = coin_flip_positions(spy, seed=1)
rows = []
for cost in [0, 5, 20]:
    result = run_backtest(spy, flips, cost_bps=cost)
    rows.append({"cost (bp)": cost, "total return": result.metrics["Total return"], "trades": int(result.metrics["Number of trades"])})
pd.DataFrame(rows).set_index("cost (bp)").style.format({"total return": "{:.1%}"})

,total return,trades
cost (bp),,
0,99.9%,661
5,3.3%,661
20,-85.8%,661


Hundreds of trades turn a modest 5 bp into a large loss, and 20 bp takes almost everything. The lesson is not that costs are scary. It is that **a strategy's costs scale with how often it trades**, so you must charge them and look at the number of trades.

Two things the engine does not charge, both worth remembering:
- **Slippage**: the price moves between the moment you decide and the moment your order fills, especially in small or fast-moving stocks.
- **Taxes and stamp duty**: UK stamp duty on buying UK company shares is 0.5%, which is 50 bp on every purchase.

**Try it:** change `seed=1` to another number. The exact returns change. Does the pattern, costs growing with trades, ever change?

## 3. A handful of trades proves almost nothing

A strategy that makes few trades gives you few examples to learn from. Suppose a strategy has **no skill**: each trade is a coin flip, win or lose. How often does such a strategy show a **win rate of 80% or more** purely by luck? It depends on how many trades it made:

In [5]:
rng = np.random.default_rng(0)
rows = []
for trades in [5, 20, 100]:
    win_rate = rng.binomial(trades, 0.5, 100_000) / trades
    rows.append({"number of trades": trades, "share of no-skill strategies with a win rate of 80% or more": (win_rate >= 0.8).mean()})
pd.DataFrame(rows).set_index("number of trades").style.format("{:.1%}")

,share of no-skill strategies with a win rate of 80% or more
number of trades,
5,18.5%
20,0.6%
100,0.0%


With 5 trades, roughly one no-skill strategy in five shows a win rate of 80% or more. With 100 trades almost none do. In session 5 you will meet a real strategy that makes only a handful of trades in ten years. Its win rate will look good, and now you know how little that proves.

**Rule of thumb:** count the trades before you believe anything else about a backtest.

## 4. Train and test: the exam you did not study for

If you adjust a strategy while looking at the results, you are fitting it to the past, including its noise. The honest test is to **tune on one period** (the *training* period) and then **score on a later period you did not look at** (the *test* period).

Here is the problem on a grand scale. `random_strategies` makes 400 strategies with no skill: each one switches between holding SPY and cash at random times. We score every one on **2016 to 2021** and again on **2022 onwards**, and sort by the first score.

In [6]:
from backtester.honesty import random_strategies, best_of_many

many = random_strategies(spy, n=400, seed=11)
results = best_of_many(spy, many, split="2021-12-31")
results.head(5)

,strategy,before Sharpe,after Sharpe
0,random_089,1.7034,0.7468
1,random_183,1.4660,0.8338
2,random_017,1.4633,1.2806
3,random_237,1.4474,1.1547
4,random_181,1.4249,0.2527


The top rows have impressive "before" scores. Now the exam: how did the best 20 do in the later period, compared with all 400?

In [7]:
top = results.head(20)
print(f"Best 20 before 2022, average Sharpe:   {top['before Sharpe'].mean():.2f}")
print(f"The same 20 from 2022 on:              {top['after Sharpe'].mean():.2f}")
print(f"All 400 strategies from 2022 on:       {results['after Sharpe'].mean():.2f}")
print(f"Best single strategy before, then after: {results.iloc[0]['before Sharpe']:.2f}, then {results.iloc[0]['after Sharpe']:.2f}")

Best 20 before 2022, average Sharpe:   1.39
The same 20 from 2022 on:              0.51
All 400 strategies from 2022 on:       0.51
Best single strategy before, then after: 1.70, then 0.75


As a group, the winners on old data are no better than the crowd on new data, because their old score was luck. Any one of them may do well again by chance, and that is what makes luck so persuasive. **A good score on data you tuned on tells you almost nothing.**

## 5. Trying many things manufactures luck

Notice that you did not need a clever strategy to find a "great" one. You only needed to try 400. The more things you try, the better the best one looks, even when none has any skill. The cell below tries 1, 5, 20, 100 and 400 strategies, 100 times each, and reports the average winner's score before and after.

In [8]:
rows = []
for tried in [1, 5, 20, 100, 400]:
    winners = [results.sample(tried, random_state=seed).sort_values("before Sharpe").iloc[-1] for seed in range(100)]
    rows.append({
        "strategies tried": tried,
        "winner's Sharpe, before": np.mean([w["before Sharpe"] for w in winners]),
        "same strategy, after": np.mean([w["after Sharpe"] for w in winners]),
    })
pd.DataFrame(rows).set_index("strategies tried").style.format("{:.2f}")

,"winner's Sharpe, before","same strategy, after"
strategies tried,,
1,0.78,0.52
5,1.13,0.56
20,1.32,0.58
100,1.50,0.78
400,1.70,0.75


The "before" column climbs with every extra strategy tried. The "after" column moves far less. Searching for longer made the past look better, not the future. Remember this whenever someone shows you the best of many tries: **ask how many they tried.** Every student who adjusts a strategy "just once more" is trying another one.

## 6. Walk-forward: repeat the exam, honestly

A single split can be lucky or unlucky too. A **walk-forward** test repeats the exam many times: tune on three years, test on the next year, slide everything forward a year, and repeat. The test years never overlap, and each one uses only the past.

To have something to tune we use a plain trend rule: *hold SPY while its price is above its average over the last N days.* The walk-forward chooses N from 20, 50, 100 or 200 using only the three years before each test year.

In [9]:
from backtester.honesty import walk_forward

def trend(prices, days):
    return (prices["Close"] > prices["Close"].rolling(days).mean()).astype(int)

folds = walk_forward(spy, trend, [{"days": d} for d in [20, 50, 100, 200]], train_days=756, test_days=252)
folds.style.format({
    "train from": "{:%Y-%m-%d}", "test from": "{:%Y-%m-%d}", "test to": "{:%Y-%m-%d}",
    "train Sharpe": "{:.2f}", "test Sharpe": "{:.2f}", "test return": "{:.1%}", "buy and hold test return": "{:.1%}",
})

,train from,test from,test to,chosen,train Sharpe,test Sharpe,test return,buy and hold test return
0,2016-01-04,2019-01-04,2020-01-03,days=200,0.92,1.44,16.5%,30.2%
1,2017-01-03,2020-01-06,2021-01-04,days=20,1.29,1.58,27.4%,16.1%
2,2018-01-03,2021-01-05,2022-01-03,days=20,1.34,1.28,12.0%,30.4%
3,2019-01-04,2022-01-04,2023-01-04,days=20,1.77,-1.27,-17.0%,-18.3%
4,2020-01-06,2023-01-05,2024-01-05,days=100,0.61,1.36,15.7%,25.2%
5,2021-01-05,2024-01-08,2025-01-07,days=200,0.69,1.88,25.6%,25.6%
6,2022-01-04,2025-01-08,2026-01-09,days=200,0.63,1.15,12.8%,19.1%


Look at the "train Sharpe" and "test Sharpe" columns side by side. The test scores jump around: some years beat the training score, and one year is badly negative. A single year is a small sample, so one year's Sharpe ratio is noisy. That is why the walk-forward matters: it gives you several fair exams instead of one.

Join the test years together to see what you could actually have earned by following the process, and compare it with buy and hold over the same years:

In [10]:
stitched = (1 + folds["test return"]).prod() - 1
held = (1 + folds["buy and hold test return"]).prod() - 1
print(f"Walk-forward, test years joined: {stitched:.1%}")
print(f"Buy and hold, same years:        {held:.1%}")

Walk-forward, test years joined: 126.0%
Buy and hold, same years:        201.4%


In this data, following the process earned less than simply holding. That is a normal, honest answer, and it is the number to compare with whatever a flattering tuning table showed you.

## 7. Two more traps, in words

**Survivorship bias.** Our data includes AAPL, MSFT and AMZN because they are giants today. Testing on today's winners flatters any strategy: nobody in 2016 knew which companies would thrive, and the ones that failed are missing from the file. Professional databases keep delisted companies. Free data usually does not.

**Hidden assumptions.** Two honest backtesters can give different numbers for the same strategy. The usual differences: when the trade fills (at the signal day's close, or at the next day's open), whether you can only buy whole shares, when the first trade happens after a warm-up period, and how costs are charged. Notebook `appendix_library_comparison` runs the same strategy in a popular library and shows exactly where the numbers drift. **Always know your backtester's assumptions.**

## 8. Seven questions to ask of any backtest

| Question | What a bad answer looks like |
|---|---|
| Could the strategy have known this at the time? | It uses a number that was only available later. |
| Were costs charged? How many trades were there? | No costs, or hundreds of trades with tiny costs. |
| Is there enough data? | A handful of trades. |
| Was it tested on data it was not tuned on? | Only one period, the one used to pick the settings. |
| How many versions were tried? | "I tried a few things" without a count. |
| Did it test only today's winners? | Survivorship. |
| Compared with what? | No benchmark, or a flattering one. |

## 9. ✏️ Your turn

1. In the costs cell, add `50` to the list of costs. What happens?
2. In section 4, change `n=400` to `n=40` and re-run sections 4 and 5. Does the winner still look as lucky?
3. In the walk-forward cell, change `train_days=756` to `504`. Do the chosen settings change?

**Check yourself:**
- A friend shows you a strategy with a Sharpe ratio of 2.0 on ten years of data and 12 trades. Which of the seven questions would you ask first, and why?
- Why does a walk-forward result usually look less impressive than the best row of a tuning table, and why is it more believable?
- You test 30 versions of a strategy and the best one looks good. What would you do before trusting it?

> Educational use only. Simulated results on historical data; not financial advice.

### Next: managing risk

Turn plain-English limits like "never fall more than 15%" into numbers and test three ways to react to a crash.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kolyamkl/qm-fintech-quant-workshop/blob/main/notebooks/03_risk_limits_one_asset.ipynb) `03_risk_limits_one_asset.ipynb`